# Galaxy rotation curves and dark matter

*cosmology · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/dark_matter_rotation)

Stars and gas far from a galaxy's centre orbit too fast: real rotation curves from the SPARC database stay flat where the visible mass predicts a Keplerian fall. One batched autograd fit, all galaxies at once, compares four explanations: baryons alone (with a mass-to-light ratio $\Upsilon$), baryons plus an NFW dark-matter halo, baryons plus a cored Burkert halo, and MOND with a single acceleration $a_0$ (the radial acceleration relation). Dark matter is inferred from gravity, not directly detected.

**Vector form**

$$V^2(r) = V_\mathrm{gas}^2 + \Upsilon_d V_\mathrm{disk}^2 + \Upsilon_b V_\mathrm{bul}^2 + V_\mathrm{halo}^2,\qquad V^2_\mathrm{NFW} = V_s^2\,\frac{\ln(1+x) - \frac{x}{1+x}}{x},\qquad g_\mathrm{obs} = \frac{g_\mathrm{bar}}{1 - e^{-\sqrt{g_\mathrm{bar}/a_0}}},\qquad \chi^2 = \sum_r \Big(\frac{V_r - V(r)}{\sigma_r}\Big)^2$$

**Index (tensor) form**

$$V^2_{gr} = V^2_{\mathrm{gas},gr} + \Upsilon_{d,g} V^2_{\mathrm{disk},gr} + \Upsilon_{b,g} V^2_{\mathrm{bul},gr} + V^2_{\mathrm{halo}}(r_{gr};\,V_{s,g}, r_{s,g}),\qquad \chi^2_{mg} = \sum_r w_{gr}\Big(\frac{V_{gr} - V_{mgr}}{\sigma_{gr}}\Big)^2,\qquad L = \sum_{m,g}\chi^2_{mg} + \text{prior}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_{gr},\ \sigma_{gr}$ | `load_sparc(device)` | (G, R) | observed speeds and errors of galaxy $g$ at radius $r$, padded to the longest curve |
| $w_{gr}$ | `data['mask']` | (G, R) | real data point (1) or padding (0) |
| $V^2_{\mathrm{bar},gr}$ | `baryonic_v2(gas, disk, bul, ups_d, ups_b)` | (G, R) | squared speeds add; $\Upsilon$ rescales the SPARC disk and bulge (computed for $\Upsilon = 1$) |
| $V^2_\mathrm{NFW}$ | `nfw_velocity(R, V_s, r_s) ** 2` | (G, R) | cuspy halo from cold-dark-matter simulations, $\rho \propto 1/(x(1+x)^2)$ |
| $V^2_\mathrm{Bur}$ | `burkert_velocity(R, V_0, r_0) ** 2` | (G, R) | cored halo, $\rho \propto 1/((1+x)(1+x^2))$ |
| $g_\mathrm{obs}(g_\mathrm{bar})$ | `mond_acceleration(V2_bar / R, a0)` | (G, R) | one function with one $a_0$ for all galaxies; no halo |
| $\chi^2_{mg}$ | `chi2(model_v2(m, p, data), data)` | (G,) | repeated $r$ ⇒ summed over the real points of each galaxy |
| $L$ | `sum(... for m in MODELS)` | () | one scalar for all models and galaxies: one backward pass gives every gradient (Adam, then L-BFGS) |

Data: six galaxies (NGC 3198, NGC 2403, NGC 6503, DDO 154, UGC 2885, NGC 7814) from SPARC (Lelli, McGaugh and Schombert 2016, AJ 152, 157), file Rotmod_LTG, copied unchanged into data/sparc_rotation_curves.csv. Source http://astroweb.cwru.edu/SPARC/, which asks users to cite the SPARC paper; a mirror on Zenodo (doi:10.5281/zenodo.16284118) is released under CC-BY-4.0. Mass-to-light ratios at 3.6 μm carry a 0.1 dex prior around $\Upsilon_d = 0.5$, $\Upsilon_b = 0.7$ (set the prior to 0 to switch it off). What the fit shows: no sensible $\Upsilon$ lets the baryons reach the flat outer speeds (DDO 154 would need $\Upsilon_d \approx 19$, far beyond any stellar population), so either unseen mass or modified dynamics is needed, as Rubin and Ford found from optical spectra in the 1970s. Both halo shapes fit well; the dwarf DDO 154 prefers a core (the cusp–core problem). A single $a_0 \approx 1.1\times10^{-10}$ m/s² describes all galaxies surprisingly well, as the radial acceleration relation says. Honest status: dark matter is inferred from its gravity (rotation curves, lensing, clusters, the CMB, structure formation); no dark-matter particle has been detected directly. MOND fits galaxy rotation curves well with one parameter but has trouble with galaxy clusters, the Bullet Cluster and the CMB acoustic peaks, and has no generally accepted relativistic version. Distances and inclinations are fixed here (their errors are not propagated), and MOND is applied in its simplest algebraic form.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/dark_matter_rotation/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    data = tensors.load_sparc(dev)
    data["prior_dex"] = params["prior_dex"] if params["prior_dex"] > 0 else None
    fit = solve.fit_all(data, iterations=params["iterations"], lr=params["lr"])
    R_fine, curves = solve.model_curves(fit, data)

    print(f"SPARC (Lelli, McGaugh & Schombert 2016): {len(data['names'])} galaxies, {int(data['mask'].sum())} points")
    print(f"one batched fit: 4 models × {len(data['names'])} galaxies, Υ prior ±{params['prior_dex']} dex")
    print(f"{'galaxy':10s} " + " ".join(f"{m:>9s}" for m in solve.MODELS) + "   (reduced χ²)")
    for g, name in enumerate(data["names"]):
        print(f"{name:10s} " + " ".join(f"{float(fit[m]['chi2_nu'][g]):9.2f}" for m in solve.MODELS))
    for g, name in enumerate(data["names"]):
        print(f"{name:10s} baryons-only needs Υ_disk = {float(fit['baryons']['ups_d'][g]):5.2f}   "
              f"NFW: Υ_d = {float(fit['nfw']['ups_d'][g]):.2f}, r_s = {float(fit['nfw']['r_s'][g]):6.1f} kpc   "
              f"Burkert: r_0 = {float(fit['burkert']['r_s'][g]):5.1f} kpc")  # fmt: skip
    a0 = float(fit["mond"]["a0"]) * tensors.MS2_PER_KMS2_KPC
    print(f"MOND / RAR: one a₀ for all galaxies = {a0:.3e} m/s² (McGaugh et al. 2016: 1.20e-10)")
    print("Dark matter is inferred from gravity, not directly detected; MOND fits galaxies but struggles with")
    print("clusters and the CMB.")
    return plot.animate(data, fit, R_fine, curves, out_dir / "dark_matter_rotation.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/dark_matter_rotation/equation.py
r"""Physics → tensor translation: circular speeds of baryons and halos, and MOND.

In a disk galaxy the circular speed at radius r balances gravity, V²/r = g(r). Independent mass components add
their accelerations, so their squared speeds add (in quadrature):

    V²_{gr} = V²_gas + Υ_d V²_disk + Υ_b V²_bulge + V²_halo           (g = galaxy, r = radius)

SPARC gives V_disk and V_bulge for a mass-to-light ratio Υ = 1, so Υ just rescales V². Halo models:

    NFW:      ρ = ρ_s / (x (1+x)²),        V² = V_s² [ln(1+x) − x/(1+x)] / x,             x = r/r_s, V_s² = 4πGρ_s r_s²
    Burkert:  ρ = ρ_0 / ((1+x)(1+x²)),     V² = V_0² [ln(1+x²) + 2 ln(1+x) − 2 arctan x] / x,  V_0² = πGρ_0 r_0²

MOND / radial acceleration relation (McGaugh, Lelli & Schombert 2016): no halo, but the observed acceleration is a
universal function of the Newtonian baryonic one,  g_obs = g_bar / (1 − e^{−√(g_bar/a₀)}),  so for g_bar ≪ a₀
g_obs → √(g_bar a₀) and V⁴ = G M a₀ (the baryonic Tully–Fisher relation).
"""

import torch

G_NEWTON = 4.30091e-6  # kpc (km/s)² / M_sun


def baryonic_v2(gas, disk, bul, ups_d, ups_b):
    """V²_bar = V_gas|V_gas| + Υ_d V_disk|V_disk| + Υ_b V_bul|V_bul|, (G, R). Υ: (G,). SPARC signs V < 0 where a
    component pulls outwards (a central gas hole), so V|V| keeps that sign."""
    return gas * gas.abs() + ups_d[:, None] * disk * disk.abs() + ups_b[:, None] * bul * bul.abs()


def nfw_velocity(r, V_s, r_s):
    """Circular speed of an NFW halo, V² = V_s² [ln(1+x) − x/(1+x)]/x with x = r/r_s."""
    x = r / r_s
    return V_s * torch.sqrt((torch.log1p(x) - x / (1 + x)) / x)


def burkert_velocity(r, V0, r0):
    """Circular speed of a Burkert (cored) halo, V² = V_0² [ln(1+x²) + 2 ln(1+x) − 2 arctan x]/x, x = r/r_0."""
    x = r / r0
    return V0 * torch.sqrt((torch.log1p(x**2) + 2 * torch.log1p(x) - 2 * torch.atan(x)) / x)


def point_mass_velocity(r, M):
    """Kepler: V = √(G M / r)."""
    return torch.sqrt(G_NEWTON * M / r)


def exponential_disk_velocity(R, mass, Rd):
    r"""Thin exponential disk Σ = Σ_0 e^{−R/R_d} (Freeman 1970): V² = 4πGΣ_0 R_d y² [I_0K_0 − I_1K_1](y), y = R/2R_d.

    Exponentially scaled Bessel functions (i0e · k0e = I_0 K_0) keep the products finite at large y.
    """
    sigma0 = mass / (2 * torch.pi * Rd**2)
    y = R / (2 * Rd)
    bessel = torch.special.i0e(y) * torch.special.scaled_modified_bessel_k0(y)
    bessel = bessel - torch.special.i1e(y) * torch.special.scaled_modified_bessel_k1(y)
    return torch.sqrt(4 * torch.pi * G_NEWTON * sigma0 * Rd * y**2 * bessel)


def total_velocity(V2_bar, V2_halo):
    """Squares add: V = √(V²_bar + V²_halo)."""
    return torch.sqrt((V2_bar + V2_halo).clamp(min=0))


def mond_acceleration(g_bar, a0):
    """Radial acceleration relation g_obs = g_bar / (1 − exp(−√(g_bar/a₀))) (McGaugh et al. 2016)."""
    g = g_bar.clamp(min=1e-6)
    return g / (-torch.expm1(-torch.sqrt(g / a0)))


def model_v2(model: str, p: dict, data: dict):
    """Model V² for every galaxy and radius, (G, R). p holds the positive parameters of that model."""
    R = data["R"]
    V2_bar = baryonic_v2(data["gas"], data["disk"], data["bul"], p["ups_d"], p["ups_b"])
    if model == "baryons":
        return V2_bar
    if model == "nfw":
        return V2_bar + nfw_velocity(R, p["V_s"][:, None], p["r_s"][:, None]) ** 2
    if model == "burkert":
        return V2_bar + burkert_velocity(R, p["V_s"][:, None], p["r_s"][:, None]) ** 2
    return mond_acceleration(V2_bar / R, p["a0"]) * R  # MOND: V² = g_obs r


def chi2(V2_model, data):
    """χ²_g = Σ_r ((V_obs − V_model)/σ)² over the real points of each galaxy, (G,)."""
    resid = (data["V"] - total_velocity(V2_model, torch.zeros_like(V2_model))) / data["err"]
    return (resid**2 * data["mask"]).sum(1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/dark_matter_rotation/tensors.py
"""Build the concrete tensors: the SPARC rotation curves padded into (galaxy, radius) arrays, and synthetic galaxies.

Units: kpc, km/s, solar masses; accelerations in (km/s)²/kpc (1 (km/s)²/kpc = 3.24 × 10⁻¹⁴ m/s²).
"""

from pathlib import Path

import torch

from . import equation
from .equation import G_NEWTON  # noqa: F401  kpc (km/s)² / M_sun

F64 = torch.float64
MS2_PER_KMS2_KPC = 3.24078e-14  # 1 (km/s)²/kpc in m/s²
A0_DEFAULT = 1.2e-10 / MS2_PER_KMS2_KPC  # Milgrom's a₀ = 1.2 × 10⁻¹⁰ m/s² ≈ 3700 (km/s)²/kpc
UPS_DISK, UPS_BULGE = 0.5, 0.7  # mass-to-light ratios at 3.6 μm expected from stellar populations (Lelli+ 2016)
DATA = Path(__file__).parent / "data" / "sparc_rotation_curves.csv"
FIELDS = ["R", "V", "err", "gas", "disk", "bul"]


def load_sparc(device: torch.device, path: Path = DATA) -> dict:
    """Read the CSV (plain Python; # lines are comments) and pad every galaxy to the longest curve.

    Returns tensors of shape (G, Rmax) for R, V, err, gas, disk, bul, a boolean mask of real points, the names and
    the distances. Padding uses R = 1, err = 1 and zero speeds, and is switched off by the mask.
    """
    rows: dict[str, list[list[float]]] = {}
    dist: dict[str, float] = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or line.startswith("galaxy") or not line.strip():
                continue
            name, d, *vals = line.strip().split(",")
            rows.setdefault(name, []).append([float(v) for v in vals])
            dist[name] = float(d)
    names = list(rows)
    n = max(len(r) for r in rows.values())
    out = {k: torch.zeros(len(names), n, dtype=F64) for k in FIELDS}
    out["R"] += 1.0
    out["err"] += 1.0
    mask = torch.zeros(len(names), n, dtype=torch.bool)
    for g, name in enumerate(names):
        block = torch.tensor(rows[name], dtype=F64)  # (N_g, 6)
        for k, field in enumerate(FIELDS):
            out[field][g, : len(block)] = block[:, k]
        mask[g, : len(block)] = True
    data = {k: v.to(device) for k, v in out.items()}
    data.update(mask=mask.to(device), names=names, distance=[dist[n] for n in names])
    data["has_bulge"] = (data["bul"].abs().amax(1) > 0).to(F64)
    return data


def synthetic_galaxies(device: torch.device, noise: float = 0.0, seed: int = 0) -> tuple[dict, dict]:
    """Three mock galaxies: exponential stellar disk (Υ_d = 1 template) plus gas plus an NFW halo, known parameters."""
    truth = {
        "ups_d": torch.tensor([0.5, 0.65, 0.4], dtype=F64),
        "V_s": torch.tensor([160.0, 90.0, 240.0], dtype=F64),
        "r_s": torch.tensor([12.0, 5.0, 20.0], dtype=F64),
    }
    M_d = torch.tensor([4e10, 5e9, 1.2e11], dtype=F64)[:, None]
    Rd = torch.tensor([3.0, 1.5, 5.0], dtype=F64)[:, None]
    R = torch.linspace(0.3, 1.0, 30, dtype=F64)[None, :] * 10 * Rd  # out to 10 R_d, (3, 30)
    disk = equation.exponential_disk_velocity(R, M_d, Rd)
    gas = 0.5 * equation.exponential_disk_velocity(R, M_d, 3 * Rd)
    V2 = (
        gas**2
        + truth["ups_d"][:, None] * disk**2
        + equation.nfw_velocity(R, truth["V_s"][:, None], truth["r_s"][:, None]) ** 2
    )
    V = V2.sqrt()
    err = 0.03 * V + 2.0
    gen = torch.Generator().manual_seed(seed)
    V = V + noise * err * torch.randn(V.shape, generator=gen, dtype=F64)
    data = {"R": R, "V": V, "err": err, "gas": gas, "disk": disk, "bul": torch.zeros_like(R)}
    data = {k: v.to(device) for k, v in data.items()}
    data.update(mask=torch.ones_like(R, dtype=torch.bool, device=device), names=["mock A", "mock B", "mock C"],
                has_bulge=torch.zeros(3, dtype=F64, device=device), prior_dex=None)  # fmt: skip
    return data, {k: v.to(device) for k, v in truth.items()}


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/dark_matter_rotation/solve.py
"""One batched autograd fit: four models × all galaxies, minimised together (Adam, then L-BFGS).

The four models share no parameters, so the sum of their χ² is minimised exactly when each is; MOND's a₀ is one
number for all galaxies. Parameters live in log space (they are all positive). Mass-to-light ratios carry a
log-normal prior of 0.1 dex around the stellar-population values Υ_d = 0.5, Υ_b = 0.7 (as in SPARC halo fits).
"""

import math

import torch

from problems._common.progress import steps

from . import equation, tensors

MODELS = ["baryons", "nfw", "burkert", "mond"]
LOG_RS = (math.log(0.3), math.log(300.0))  # halo scale radius kept between 0.3 and 300 kpc
LOG_VS = (math.log(5.0), math.log(2000.0))


def initial_parameters(data: dict) -> dict:
    """Raw (log) parameters for every model, each tensor (G,) except log a₀ (scalar)."""
    G = data["R"].shape[0]
    V_max = (data["V"] * data["mask"]).amax(1)
    R_max = (data["R"] * data["mask"]).amax(1)
    ones = torch.ones(G, dtype=tensors.F64, device=data["R"].device)

    def base():
        return {"ups_d": ones * math.log(tensors.UPS_DISK), "ups_b": ones * math.log(tensors.UPS_BULGE)}

    raw = {m: base() for m in MODELS}
    raw["nfw"].update(V_s=torch.log(2.0 * V_max), r_s=torch.log(0.5 * R_max))
    raw["burkert"].update(V_s=torch.log(1.5 * V_max), r_s=torch.log(0.3 * R_max))
    raw["mond"]["a0"] = torch.tensor(math.log(tensors.A0_DEFAULT), dtype=tensors.F64, device=data["R"].device)
    return {m: {k: v.clone().requires_grad_(True) for k, v in p.items()} for m, p in raw.items()}


def physical(raw: dict) -> dict:
    """exp of the raw parameters, with the halo parameters clamped to their ranges."""
    p = {}
    for k, v in raw.items():
        if k == "r_s":
            v = v.clamp(*LOG_RS)
        elif k == "V_s":
            v = v.clamp(*LOG_VS)
        p[k] = torch.exp(v)
    return p


def prior(raw: dict, data: dict, dex: float | None) -> torch.Tensor:
    """Σ_g ((log₁₀Υ − log₁₀Υ₀)/dex)² for disks and (where present) bulges."""
    if dex is None:
        return torch.zeros((), dtype=tensors.F64, device=data["R"].device)
    ld = (raw["ups_d"] - math.log(tensors.UPS_DISK)) / math.log(10) / dex
    lb = (raw["ups_b"] - math.log(tensors.UPS_BULGE)) / math.log(10) / dex
    return (ld**2).sum() + (lb**2 * data["has_bulge"]).sum()


def fit_all(data: dict, iterations: int = 800, lr: float = 0.05) -> dict:
    """Fit all models to all galaxies. Returns, per model, the parameters, χ² (G,) and reduced χ² (G,)."""
    raw = initial_parameters(data)
    dex = data.get("prior_dex", 0.1)
    params = [v for p in raw.values() for v in p.values()]

    def objective():
        return sum(equation.chi2(equation.model_v2(m, physical(raw[m]), data), data).sum() + prior(raw[m], data, dex)
                   for m in MODELS)  # fmt: skip

    opt = torch.optim.Adam(params, lr=lr)
    for _ in steps(iterations, "Adam"):  # robust first descent
        opt.zero_grad()
        objective().backward()
        opt.step()
    polish = torch.optim.LBFGS(params, max_iter=300, tolerance_grad=1e-9, line_search_fn="strong_wolfe")

    def closure():
        polish.zero_grad()
        loss = objective()
        loss.backward()
        return loss

    polish.step(closure)  # quasi-Newton to the bottom of the valley

    n = data["mask"].sum(1).to(tensors.F64)
    k_bar = 1 + data["has_bulge"]  # Υ_d (and Υ_b)
    n_par = {"baryons": k_bar, "nfw": k_bar + 2, "burkert": k_bar + 2, "mond": k_bar}
    out = {}
    with torch.no_grad():
        for m in MODELS:
            p = physical(raw[m])
            c2 = equation.chi2(equation.model_v2(m, p, data), data)
            out[m] = {**p, "chi2": c2, "chi2_nu": c2 / (n - n_par[m]).clamp(min=1)}
    return out


def model_curves(fit: dict, data: dict, n: int = 120) -> tuple[torch.Tensor, dict]:
    """Smooth model curves on a fine radius grid: SPARC baryon speeds interpolated linearly, (G, n) each."""
    R_max = (data["R"] * data["mask"]).amax(1, keepdim=True)
    R_min = data["R"][:, :1]  # first measured radius (the baryon models are only known where SPARC measured)
    s = torch.linspace(0, 1, n, dtype=tensors.F64, device=R_max.device)[None, :]
    R = R_min + s * (R_max - R_min)  # (G, n)
    fine = {"R": R, "mask": torch.ones_like(R, dtype=torch.bool)}
    for k in ("gas", "disk", "bul"):
        fine[k] = torch.stack([_interp(R[g], data["R"][g][data["mask"][g]], data[k][g][data["mask"][g]])
                               for g in range(R.shape[0])])  # fmt: skip
    curves = {m: equation.total_velocity(equation.model_v2(m, fit[m], fine), torch.zeros_like(R)) for m in MODELS}
    return R, curves


def _interp(x, xp, fp):
    """Linear interpolation (flat beyond the ends), like numpy.interp."""
    k = torch.searchsorted(xp, x).clamp(1, len(xp) - 1)
    w = ((x - xp[k - 1]) / (xp[k] - xp[k - 1])).clamp(0, 1)
    return fp[k - 1] * (1 - w) + fp[k] * w


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/dark_matter_rotation/plot.py
"""Animate the fits galaxy by galaxy, while the radial acceleration relation fills up point by point."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation
from .tensors import MS2_PER_KMS2_KPC

LABELS = {
    "baryons": "baryons only",
    "nfw": "baryons + NFW halo",
    "burkert": "baryons + Burkert core",
    "mond": "MOND (RAR)",
}
COLOURS = {"baryons": "#6b7280", "nfw": ACCENT[0], "burkert": ACCENT[2], "mond": ACCENT[1]}


def animate(data: dict, fit: dict, R_fine, curves: dict, path: Path, per_galaxy: int = 14, tail: int = 10) -> Path:
    names = data["names"]
    mask = data["mask"].cpu().numpy()
    R, V, err = (data[k].cpu().numpy() for k in ("R", "V", "err"))
    Rf = R_fine.cpu().numpy()
    cv = {m: c.cpu().numpy() for m, c in curves.items()}
    chi = {m: fit[m]["chi2_nu"].cpu().numpy() for m in curves}
    # radial acceleration relation with the MOND-fit mass-to-light ratios: g_obs = V²/R against g_bar = V²_bar/R
    V2_bar = equation.baryonic_v2(data["gas"], data["disk"], data["bul"], fit["mond"]["ups_d"], fit["mond"]["ups_b"])
    g_bar = (V2_bar / data["R"]).cpu().numpy() * MS2_PER_KMS2_KPC
    g_obs = (data["V"] ** 2 / data["R"]).cpu().numpy() * MS2_PER_KMS2_KPC
    a0 = float(fit["mond"]["a0"]) * MS2_PER_KMS2_KPC

    fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(9.6, 4.4), gridspec_kw={"width_ratios": [1.2, 1]})
    pts = ax.errorbar([], [], yerr=[], fmt="o", ms=3, color="#111827", ecolor="#9ca3af", lw=1, label="observed (SPARC)")
    lines = {m: ax.plot([], [], lw=2, color=COLOURS[m])[0] for m in curves}
    ax.set_xlabel("radius r (kpc)")
    ax.set_ylabel("rotation speed V (km/s)")

    gx = np.logspace(-12.3, -8.5, 200)
    ax_r.set_xscale("log")
    ax_r.set_yscale("log")
    ax_r.set_xlim(gx[0], gx[-1])
    ax_r.set_ylim(gx[0], gx[-1])
    ax_r.plot(gx, gx, "--", color="#9ca3af", lw=1, label="g_obs = g_bar (no dark matter)")
    rar = ax_r.plot([], [], lw=2, color=ACCENT[1], label="")[0]
    scat = [ax_r.plot([], [], "o", ms=2.5, alpha=0.7, color=ACCENT[g % len(ACCENT)])[0] for g in range(len(names))]
    ax_r.set_xlabel("baryonic (Newtonian) acceleration $g_{bar}$ (m/s²)")
    ax_r.set_ylabel("observed $g_{obs} = V^2/r$ (m/s²)")
    ax_r.set_title("radial acceleration relation", fontsize=10)
    ax_r.legend(fontsize=7, loc="upper left")

    def update(frame: int):
        g = min(frame // per_galaxy, len(names) - 1)
        k = frame - g * per_galaxy if frame < len(names) * per_galaxy else per_galaxy
        m_ = mask[g]
        nonlocal pts
        pts.remove()
        pts = ax.errorbar(R[g, m_], V[g, m_], yerr=err[g, m_], fmt="o", ms=3, color="#111827", ecolor="#9ca3af",
                          lw=1, label="observed (SPARC)")  # fmt: skip
        top = 1.25 * max(V[g, m_].max(), max(cv[m][g].max() for m in cv))
        ax.set_xlim(0, 1.05 * Rf[g, -1])
        ax.set_ylim(0, top)
        ax.set_title(f"{names[g]}  (D = {data['distance'][g]} Mpc)", fontsize=10)
        for j, m in enumerate(cv):  # curve j grows during frames 3 + 3j … 5 + 3j
            share = np.clip((k - 2 - 3 * j) / 3, 0, 1)
            n = int(round(share * Rf.shape[1]))
            lines[m].set_data(Rf[g, :n], cv[m][g, :n])
            lines[m].set_label(f"{LABELS[m]}  χ²/ν = {chi[m][g]:.1f}" if n else LABELS[m])
        ax.legend(fontsize=7, loc="upper left")
        for h in range(len(names)):
            show = h < g or (h == g and k >= 2)
            scat[h].set_data(g_bar[h, mask[h]] if show else [], g_obs[h, mask[h]] if show else [])
        if frame >= len(names) * per_galaxy:
            rar.set_data(gx, gx / (1 - np.exp(-np.sqrt(gx / a0))))
            rar.set_label(f"fit: $a_0$ = {a0:.2e} m/s²")
            ax_r.legend(fontsize=7, loc="upper left")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(names) * per_galaxy + tail), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'iterations': 500,
    'lr': 0.05,
    'prior_dex': 0.1
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.dark_matter_rotation")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.dark_matter_rotation.main")
problem_dir = Path("problems/cosmology/dark_matter_rotation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
